# Patient-Speech-Only Poisson GLM Encoding Orchestrator

Identical analysis to `word_level_duration_cv_all_n.ipynb`, but restricted to words that are:

1. **Quality-filtered** (`YFS_word_df_filtered.csv` — audio quality passes all metric fences)
2. **Patient-speaking** (`YFS_transcripts_annotated.csv` — LR-ASD diarization marks segment as patient speech)

Word subset indices are saved per-patient and passed to the same `poisson_glm_worker.py` via `--word-idx-path`.

### 1. Imports

In [1]:
from pathlib import Path
import subprocess

import dill as pickle
import numpy as np
import pandas as pd

### 2. Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PROJ_DIR    = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT    = PROJ_DIR / 'vad_new'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                   '/standard_encoding_analysis/poisson_glm_worker.py')
PYTHON_BIN  = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'

PATIENTS       = None   # None => scan all Y* patient folders
FORCE_RESUBMIT = False

# ── Model settings ────────────────────────────────────────────────────────────
SPIKE_OFFSET_IDX = 0
GPT2_LAYER       = -1
N_PCA            = 100
OUTER_SPLITS     = 5
INNER_SPLITS     = 5
N_ALPHAS         = 30
ALPHA_LOW        = -3.0
ALPHA_HIGH       = 3.0
N_SHUFFLES       = 50

# ── Compute mode — set False to submit CPU-only jobs (no GPU slot required) ───
USE_GPU = True

# ── SLURM ─────────────────────────────────────────────────────────────────────
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env'
if USE_GPU:
    PARTITION = 'Universal'
    GRES      = 'gpu:1'
    CPUS      = 8
    MEM       = '64G'
    TIME      = '24:00:00'
else:
    PARTITION = 'Universal'
    GRES      = ''       # no GPU
    CPUS      = 16
    MEM       = '32G'
    TIME      = '48:00:00'

RUN_NAME          = 'word_level_duration_cv_patient_speech'
GLOBAL_LOGS_DIR   = VAD_ROOT / f'{RUN_NAME}_slurm_logs'
GLOBAL_SCRIPTS_DIR = VAD_ROOT / f'{RUN_NAME}_slurm_scripts'
GLOBAL_LOGS_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('vad root:   ', VAD_ROOT)
print('worker:     ', WORKER_PATH)
print('logs dir:   ', GLOBAL_LOGS_DIR)
print(f'compute:     {"GPU" if USE_GPU else "CPU"} | partition={PARTITION}')

vad root:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new
worker:      /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_encoding_analysis/poisson_glm_worker.py
logs dir:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/word_level_duration_cv_patient_speech_slurm_logs
compute:     GPU | partition=Universal


### 3. Compute & Save Patient-Speech Word Indices Per Patient

In [3]:
def compute_patient_speech_indices(patient: str) -> Path | None:
    """
    Merge quality-filtered word df with patient_speaking annotations.
    Returns path to saved .npy index file, or None if inputs are missing.
    """
    patient_root = VAD_ROOT / patient
    filtered_csv   = patient_root / f'{patient}_word_df_filtered.csv'
    annotated_csv  = patient_root / f'{patient}_transcripts_annotated.csv'
    idx_out_path   = patient_root / 'encoding' / RUN_NAME / f'{patient}_patient_speech_word_idx.npy'

    if not filtered_csv.exists():
        print(f'  [{patient}] missing {filtered_csv.name}')
        return None
    if not annotated_csv.exists():
        print(f'  [{patient}] missing {annotated_csv.name} — run add_video_diarization_df.ipynb first')
        return None

    filtered_df  = pd.read_csv(filtered_csv)
    annotated_df = pd.read_csv(annotated_csv, index_col=0)

    # original_word_idx is the row index into the un-filtered transcripts CSV,
    # which is also the index of annotated_df
    patient_speaking = annotated_df['patient_speaking'].astype(bool)
    filtered_df['patient_speaking'] = patient_speaking.reindex(
        filtered_df['original_word_idx']
    ).values

    patient_words = filtered_df[filtered_df['patient_speaking'] == True]
    word_idx = patient_words['original_word_idx'].values.astype(np.int64)

    idx_out_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(idx_out_path, word_idx)

    n_total    = len(filtered_df)
    n_patient  = len(word_idx)
    print(f'  [{patient}] {n_patient:,} patient-speech words / {n_total:,} quality-filtered '
          f'({100*n_patient/n_total:.1f}%) → {idx_out_path.name}')
    return idx_out_path


# Discover patients
if PATIENTS is None:
    all_patients = sorted([p.name for p in VAD_ROOT.iterdir() if p.is_dir() and p.name.startswith('Y')])
else:
    all_patients = PATIENTS

print(f'Computing patient-speech word indices for {len(all_patients)} patients...')
patient_idx_paths = {}
for patient in all_patients:
    idx_path = compute_patient_speech_indices(patient)
    if idx_path is not None:
        patient_idx_paths[patient] = idx_path

print(f'\nReady to run: {len(patient_idx_paths)} / {len(all_patients)} patients')

Computing patient-speech word indices for 21 patients...
  [YEY] missing YEY_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YEZ] missing YEZ_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFA] missing YFA_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFB] missing YFB_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFC] 17,828 patient-speech words / 292,428 quality-filtered (6.1%) → YFC_patient_speech_word_idx.npy
  [YFD] 16,307 patient-speech words / 388,734 quality-filtered (4.2%) → YFD_patient_speech_word_idx.npy
  [YFE] 3,845 patient-speech words / 199,717 quality-filtered (1.9%) → YFE_patient_speech_word_idx.npy
  [YFF] 11,244 patient-speech words / 228,795 quality-filtered (4.9%) → YFF_patient_speech_word_idx.npy
  [YFG] 29 patient-speech words / 48,567 quality-filtered (0.1%) → YFG_patient_speech_word_idx.npy
  [YFI] 5,830 patient-speech words / 126,297 quality-filtered (4.6%) → YF

### 4. Resolve Input Paths & Build Status Table

In [6]:
def first_existing(paths):
    for path in paths:
        if path is not None and Path(path).exists():
            return Path(path)
    return None


def resolve_patient_paths(patient: str) -> dict:
    patient_root = VAD_ROOT / patient
    embeddings_path = first_existing([
        patient_root / 'embeddings' / f'{patient}_gpt2_embeddings.npy',
        patient_root / 'all_convo_recording' / 'all_words_filtered_all_layers_gpt2.npy',
    ])
    counts_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_spike_counts_offsets_all.npy',
        patient_root / 'all_convo_recording' / 'word_spike_counts_offsets_all.npy',
    ])
    durations_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_durs.npy',
        patient_root / 'all_convo_recording' / 'word_durs.npy',
    ])

    out_dir      = patient_root / 'encoding' / RUN_NAME
    result_pkl   = out_dir / f'{patient}_encoding_results_cv.pkl'
    result_tar   = out_dir / f'{patient}_encoding_models_cv.tar'
    meta_json    = out_dir / f'{patient}_meta.json'
    success_path = out_dir / f'{patient}_SUCCESS'
    error_path   = out_dir / f'{patient}_error.txt'
    idx_path     = patient_idx_paths.get(patient)

    return {
        'patient': patient, 'patient_root': patient_root,
        'embeddings_path': embeddings_path, 'counts_path': counts_path,
        'durations_path': durations_path, 'word_idx_path': idx_path,
        'out_dir': out_dir, 'result_pkl': result_pkl, 'result_tar': result_tar,
        'meta_json': meta_json, 'success_path': success_path, 'error_path': error_path,
    }


def build_status_df(patients=None) -> pd.DataFrame:
    if patients is None:
        patients = sorted([p.name for p in VAD_ROOT.iterdir() if p.is_dir() and p.name.startswith('Y')])
    rows = []
    for patient in patients:
        info = resolve_patient_paths(patient)
        row = {
            **info,
            'has_embeddings':  info['embeddings_path'] is not None,
            'has_counts':      info['counts_path'] is not None,
            'has_durations':   info['durations_path'] is not None,
            'has_word_idx':    info['word_idx_path'] is not None,
            'has_success':     info['success_path'].exists(),
            'has_error':       info['error_path'].exists(),
            'has_result_pkl':  info['result_pkl'].exists(),
        }
        row['ready_inputs'] = (row['has_embeddings'] and row['has_counts']
                               and row['has_durations'] and row['has_word_idx'])
        row['needs_submit'] = row['ready_inputs'] and (FORCE_RESUBMIT or not row['has_success'])
        rows.append(row)
    return pd.DataFrame(rows).sort_values('patient').reset_index(drop=True)


status_df = build_status_df(PATIENTS)
print('patients:     ', len(status_df))
print('ready inputs: ', int(status_df['ready_inputs'].sum()))
print('completed:    ', int(status_df['has_success'].sum()))
print('errors:       ', int(status_df['has_error'].sum()))

status_df[[
    'patient', 'has_embeddings', 'has_counts', 'has_durations',
    'has_word_idx', 'ready_inputs', 'has_success', 'has_error', 'needs_submit'
]]

patients:      21
ready inputs:  8
completed:     1
errors:        1


,patient,has_embeddings,has_counts,has_durations,has_word_idx,ready_inputs,has_success,has_error,needs_submit
0,YEY,True,True,True,False,False,False,False,False
1,YEZ,True,True,True,False,False,False,False,False
2,YFA,True,True,True,False,False,False,False,False
3,YFB,True,True,True,False,False,False,False,False
4,YFC,True,True,True,True,True,False,False,True
5,YFD,True,True,True,True,True,False,False,True
6,YFE,True,True,True,False,False,False,False,False
7,YFF,True,True,True,False,False,False,False,False
8,YFG,True,True,True,False,False,False,False,False
9,YFI,True,True,True,False,False,False,False,False


### 5. Submit SLURM Jobs

In [7]:
submitted = []
failed    = []

for _, row in status_df.iterrows():
    patient = row['patient']
    if not row['needs_submit']:
        continue

    info = resolve_patient_paths(patient)
    info['out_dir'].mkdir(parents=True, exist_ok=True)
    reset_line = (
        f"rm -f {info['success_path']} {info['error_path']}" if FORCE_RESUBMIT else 'true'
    )

    cmd = (
        f'{PYTHON_BIN} {WORKER_PATH} {patient} {VAD_ROOT} {info["out_dir"]} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--gpt2-layer {GPT2_LAYER} '
        f'--n-pca {N_PCA} '
        f'--outer-splits {OUTER_SPLITS} '
        f'--inner-splits {INNER_SPLITS} '
        f'--n-alphas {N_ALPHAS} '
        f'--alpha-low {ALPHA_LOW} '
        f'--alpha-high {ALPHA_HIGH} '
        f'--n-shuffles {N_SHUFFLES} '
        f'--embeddings-path {info["embeddings_path"]} '
        f'--counts-path {info["counts_path"]} '
        f'--durations-path {info["durations_path"]} '
        f'--word-idx-path {info["word_idx_path"]}'
    )

    gres_lines = [f'#SBATCH --gres={GRES}'] if GRES else []
    lines = [
        '#!/bin/bash',
        f'#SBATCH --job-name=glm_ps_{patient}',
        f'#SBATCH --partition={PARTITION}',
        *gres_lines,
        f'#SBATCH --cpus-per-task={CPUS}',
        f"#SBATCH --exclude=guppy-1",
        f'#SBATCH --qos=big_batch_tier',
        f'#SBATCH --mem={MEM}',
        f'#SBATCH --time={TIME}',
        f'#SBATCH --output={GLOBAL_LOGS_DIR}/{patient}_%j.out',
        f'#SBATCH --error={GLOBAL_LOGS_DIR}/{patient}_%j.err',
        '',
        'set -eo pipefail',
        '',
        CONDA_ACTIVATE,
        '',
        'echo "HOSTNAME: $(hostname)"',
        'echo "START: $(date)"',
        f'echo "PATIENT: {patient}"',
        '',
        reset_line,
        '',
        cmd,
        '',
        'echo "END: $(date)"',
    ]

    sbatch_path = GLOBAL_SCRIPTS_DIR / f'{patient}.sbatch'
    sbatch_path.write_text('\n'.join(lines))

    try:
        res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
        submitted.append((patient, res.stdout.strip()))
        print(f'submitted: {patient} -> {res.stdout.strip()}')
    except subprocess.CalledProcessError as exc:
        failed.append((patient, exc.stderr.strip()))
        print(f'FAILED SUBMISSION: {patient}\n{exc.stderr}')

print(f'submitted={len(submitted)}, failed={len(failed)}')

submitted: YFC -> Submitted batch job 233612
submitted: YFD -> Submitted batch job 233613
submitted: YFM -> Submitted batch job 233614
submitted: YFN -> Submitted batch job 233615
submitted: YFR -> Submitted batch job 233616
submitted: YFT -> Submitted batch job 233617
submitted: YFV -> Submitted batch job 233618
submitted=7, failed=0


### 6. Check Status

In [6]:
status_df = build_status_df(PATIENTS)
status_df[['patient', 'ready_inputs', 'has_success', 'has_error', 'has_result_pkl']]

,patient,ready_inputs,has_success,has_error,has_result_pkl
0,YFS,True,True,True,True
1,YFV,False,False,False,False


### 7. Inspect Errors

In [7]:
error_rows = status_df[status_df['has_error']].copy()
print(f'patients with error files: {len(error_rows)}')
for _, row in error_rows.head(5).iterrows():
    print('=' * 100)
    print(row['patient'])
    print(row['error_path'].read_text()[:4000])

patients with error files: 1
YFS
Traceback (most recent call last):
  File "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_encoding_analysis/poisson_glm_worker.py", line 646, in main
    fold_metrics, summary = run_nested_cv(
                            ^^^^^^^^^^^^^^
  File "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_encoding_analysis/poisson_glm_worker.py", line 386, in run_nested_cv
    best_alpha, alpha_ll_mean, alpha_ll_std, (init_w, init_b) = _tune_alpha_inner_cv(
                                                                ^^^^^^^^^^^^^^^^^^^^^
  File "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_encoding_analysis/poisson_glm_worker.py", line 331, in _tune_alpha_inner_cv
    model, W, b = fit_poisson_ridge_lbfgs(
                  ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_encoding_analysis/poisson_glm_worker.py", line 247, in fit_poisson_ridge_lbfgs
    optimizer.st

### 8. Load Results

In [8]:
records = []
loaded  = []

for _, row in status_df.iterrows():
    if not row['has_result_pkl']:
        continue
    with open(row['result_pkl'], 'rb') as f:
        df = pickle.load(f)
    summary_df = df[df['is_summary'] == True].copy() if 'is_summary' in df.columns else pd.DataFrame()
    if summary_df.empty:
        continue
    summary_df['patient'] = row['patient']
    records.append(summary_df)
    loaded.append(row['patient'])

if records:
    all_summary = pd.concat(records, ignore_index=True)
    print('loaded patients:', loaded)
    print('summary rows:', len(all_summary))
    display(all_summary.head())

    patient_level = all_summary.groupby('patient').agg(
        n_neurons=('neuron_idx', 'nunique'),
        pseudo_r2_mean=('pseudo_r2_mean', 'mean'),
        pearson_corr_mean=('pearson_corr_mean', 'mean'),
        spearman_corr_mean=('spearman_corr_mean', 'mean'),
    ).reset_index()
    display(patient_level.sort_values('patient'))
else:
    print('No completed patient result pickles found yet.')

loaded patients: ['YFS']
summary rows: 64


,neuron_idx,fold_id,is_summary,outer_splits,inner_splits,best_alpha,alpha_ll_mean,alpha_ll_std,ll_real,ll_null,...,pearson_corr_std,spearman_corr_mean,spearman_corr_std,edf_mean,edf_std,aic_mean,aic_std,bic_mean,bic_std,patient
0,0,NaN,True,5,5,NaN,NaN,NaN,NaN,NaN,...,0.015368,0.541467,0.010086,99.876044,1.731701,43903.985156,1091.610214,44578.799105,1097.276324,YFS
1,1,NaN,True,5,5,NaN,NaN,NaN,NaN,NaN,...,0.024682,0.615571,0.006867,98.009894,1.045127,36898.503125,629.699259,37560.708898,632.114751,YFS
2,2,NaN,True,5,5,NaN,NaN,NaN,NaN,NaN,...,0.044739,0.635099,0.013097,99.487640,0.782568,32605.446484,330.696877,33277.637104,327.314587,YFS
3,3,NaN,True,5,5,NaN,NaN,NaN,NaN,NaN,...,0.054574,0.505097,0.006284,99.463675,1.373338,31310.796484,601.117394,31982.825688,604.760517,YFS
4,4,NaN,True,5,5,NaN,NaN,NaN,NaN,NaN,...,0.021373,0.713412,0.007013,99.399142,1.724754,40920.000781,403.238489,41591.593420,401.662670,YFS


,patient,n_neurons,pseudo_r2_mean,pearson_corr_mean,spearman_corr_mean
0,YFS,64,0.021717,0.479091,0.444415


### 9. Comparison Plots — Patient Speech vs All Speech

In [9]:
from pathlib import Path
import subprocess as _sp

PLOT_SCRIPT = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                   '/standard_encoding_analysis/plot_encoding_results_comparison.py')

ALL_SPEECH_RUN = 'word_level_duration_cv_all_n'

for patient in loaded:
    ps_pkl  = VAD_ROOT / patient / 'encoding' / RUN_NAME / f'{patient}_encoding_results_cv.pkl'
    all_pkl = VAD_ROOT / patient / 'encoding' / ALL_SPEECH_RUN / f'{patient}_encoding_results_cv.pkl'

    if not all_pkl.exists():
        print(f'[{patient}] all-speech result not found, skipping comparison')
        continue

    out_dir = VAD_ROOT / patient / 'encoding' / RUN_NAME
    cmd = [
        PYTHON_BIN, str(PLOT_SCRIPT),
        str(all_pkl), str(ps_pkl),
        '--out-dir', str(out_dir),
        '--patient', patient,
    ]
    result = _sp.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'[{patient}] plot FAILED:\n{result.stderr[-500:]}')
    else:
        print(f'[{patient}] plots saved to {out_dir}')

[YFS] plots saved to /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFS/encoding/word_level_duration_cv_patient_speech
